# Project 01 (basic) — A STRIPS forward planner

**Module 07 — Theory of AI 2**

You build a **classical planner**: STRIPS states and actions, forward search
(progression) with BFS *and* with A\* under the **h_add heuristic** from the
delete relaxation (script part 1). The test case is the famous **Sussman
anomaly** of the blocks world — a small problem on which naive subgoal planners
fail, but state search does not.

The goal: to understand how planning can be formulated as search (module 06) and
how a **heuristic obtained automatically from the action description** guides the
search.

## Setup
Only the standard library. Select the kernel of the repository `.venv` (see
`SETUP.md`) and run the cells from top to bottom. Then solve **tasks 1–3** (the
`TODO` cells).

In [ ]:
import heapq, itertools
from collections import deque
from dataclasses import dataclass, field
from typing import FrozenSet, Tuple, Any
print("Libraries loaded (only the standard library).")

## Part A — STRIPS and the blocks world (given)
The representation: fluents as tuples, states as a `frozenset`, actions as
`Action(pre, add, dele)`. Progression is pure set arithmetic
$(s\setminus\mathrm{DEL})\cup\mathrm{ADD}$. Read the cells and run them.

In [ ]:
# ---- The STRIPS representation ---------------------------------------------
# A fluent is a tuple, e.g. ("on","C","A") or ("handempty",).
# A state is a frozenset of fluents (closed world: whatever is missing is false).

@dataclass(frozen=True)
class Action:
    name: str
    pre: FrozenSet
    add: FrozenSet
    dele: FrozenSet          # "del" is a Python keyword -> dele
    def __repr__(self): return self.name

def applicable(action, state):
    """Applicable if all preconditions hold in the state."""
    return action.pre <= state

def result(state, action):
    """Progression:  (s without DEL) united with ADD."""
    return frozenset((state - action.dele) | action.add)

# ---- Blocks world: instantiate the action schemas for concrete blocks -------
def blocksworld_actions(blocks):
    acts = []
    for x in blocks:
        acts.append(Action(f"PickUp({x})",
            pre=frozenset({("clear", x), ("ontable", x), ("handempty",)}),
            add=frozenset({("holding", x)}),
            dele=frozenset({("clear", x), ("ontable", x), ("handempty",)})))
        acts.append(Action(f"PutDown({x})",
            pre=frozenset({("holding", x)}),
            add=frozenset({("ontable", x), ("clear", x), ("handempty",)}),
            dele=frozenset({("holding", x)})))
        for y in blocks:
            if x == y:
                continue
            acts.append(Action(f"Stack({x},{y})",
                pre=frozenset({("holding", x), ("clear", y)}),
                add=frozenset({("on", x, y), ("clear", x), ("handempty",)}),
                dele=frozenset({("holding", x), ("clear", y)})))
            acts.append(Action(f"Unstack({x},{y})",
                pre=frozenset({("on", x, y), ("clear", x), ("handempty",)}),
                add=frozenset({("holding", x), ("clear", y)}),
                dele=frozenset({("on", x, y), ("clear", x), ("handempty",)})))
    return acts

@dataclass(frozen=True)
class PlanProblem:
    initial: FrozenSet
    goal: FrozenSet
    actions: Tuple
    def is_goal(self, state): return self.goal <= state
    def successors(self, state):
        return [(a, result(state, a)) for a in self.actions if applicable(a, state)]

# ---- The Sussman anomaly (the famous blocks world test case) ---------------
BLOCKS = ["A", "B", "C"]
s0 = frozenset({("on", "C", "A"), ("ontable", "A"), ("ontable", "B"),
                ("clear", "C"), ("clear", "B"), ("handempty",)})
goal = frozenset({("on", "A", "B"), ("on", "B", "C")})
problem = PlanProblem(s0, goal, tuple(blocksworld_actions(BLOCKS)))

def show_state(s):
    parts = []
    for f in sorted(s):
        parts.append(f[0] + ("(" + ",".join(f[1:]) + ")" if len(f) > 1 else ""))
    return "  ".join(parts)

print("Start :", show_state(s0))
print("Goal  :", show_state(goal))
print("Ground actions:", len(problem.actions))

## Part B — The search infrastructure and BFS (given)
The same `Node` as in module 06 and the generic best-first search. BFS is given
completely, as a model.

In [ ]:
# ---- Search nodes (as in module 06) ----------------------------------------
@dataclass(frozen=True)
class Node:
    state: Any
    parent: Any = None
    action: Any = None
    path_cost: float = 0.0     # = the number of actions up to here
    def plan(self):
        acts, n = [], self
        while n.parent is not None:
            acts.append(n.action); n = n.parent
        return list(reversed(acts))

print("Search nodes ready.")

In [ ]:
# ---- Forward search with BFS (given completely) ----------------------------
def bfs_plan(problem):
    node = Node(problem.initial)
    if problem.is_goal(node.state):
        return node, 0
    frontier = deque([node]); reached = {node.state}; expanded = 0
    while frontier:
        node = frontier.popleft(); expanded += 1
        for a, s2 in problem.successors(node.state):
            if s2 in reached:
                continue
            child = Node(s2, node, a, node.path_cost + 1)
            if problem.is_goal(s2):
                return child, expanded
            reached.add(s2); frontier.append(child)
    return None, expanded

node, exp = bfs_plan(problem)
print("BFS plan (", len(node.plan()), "actions ), expanded states:", exp)
for i, a in enumerate(node.plan(), 1):
    print(f"  {i}. {a}")

In [ ]:
# ---- Generic best-first search (given) -------------------------------------
def best_first_plan(problem, f):
    node = Node(problem.initial)
    counter = itertools.count()
    frontier = [(f(node.state, 0), next(counter), node)]
    reached = {problem.initial: 0}
    expanded = 0
    while frontier:
        _, _, node = heapq.heappop(frontier)
        if problem.is_goal(node.state):
            return node, expanded
        expanded += 1
        for a, s2 in problem.successors(node.state):
            g2 = node.path_cost + 1
            if s2 not in reached or g2 < reached[s2]:
                reached[s2] = g2
                child = Node(s2, node, a, g2)
                heapq.heappush(frontier, (f(s2, g2), next(counter), child))
    return None, expanded

print("best_first_plan ready — A* is the right choice of f.")

### Task 1 — the h_add heuristic (delete relaxation)
Implement `h_add`: ignore all delete lists and estimate the cost of every fluent
by a fixed-point iteration; sum over the goal fluents. See the formula in the
cell comment and script part 1.4.

In [ ]:
# ---- TASK 1: delete relaxation — the h_add heuristic -----------------------
# The idea (script part 1.4): ignore ALL delete lists. Then a fluent that has been
# achieved can never be lost, and one can estimate the cost of every fluent by a
# fixed-point iteration:
#   Delta(p) = 0                              if p is already in the state,
#   Delta(p) = min_{a: p in ADD(a)} ( 1 + sum_{q in PRE(a)} Delta(q) )  otherwise.
# h_add(state) = sum_{g in goal} Delta(g).   (inf if a goal is unreachable)
#
# Hint: start with delta = {f: 0 for f in state}. Iterate over problem.actions
# until nothing changes any more: if all preconditions have a finite cost, then
# cost_a = 1 + sum(delta[q] for q in a.pre); update delta[p] = min(...).
def h_add(state, problem):
    # TODO
    raise NotImplementedError

print("h_add(start) =", h_add(problem.initial, problem))

### Task 2 — A\* with h_add
Wire `best_first_plan` up with $f(s,g)=g+h_{\text{add}}(s)$ to get A\*.

In [ ]:
# ---- TASK 2: A* with h_add -------------------------------------------------
# A* = best_first_plan with f(s, g) = g + h_add(s, problem).
def astar_plan(problem):
    # TODO: return best_first_plan with the right f
    raise NotImplementedError

node, exp = astar_plan(problem)
print("A* plan (", len(node.plan()), "actions ), expanded states:", exp)
for i, a in enumerate(node.plan(), 1):
    print(f"  {i}. {a}")

### Task 3 — the comparison
Compare BFS and A\*(h_add): the plan length and the number of expanded states.
Both should find an optimal **6-step plan**; A\* expands fewer.

In [ ]:
# ---- TASK 3: comparing BFS and A*(h_add) -----------------------------------
# Solve the problem with both procedures, print the plan length and the number of
# expanded states, and check with an assert that both plans have the same length (=6).
# Name the A* result node `a_node` — part C uses it for the verification.
# TODO
raise NotImplementedError

## Part C — Verification
Finally we check that the plan found is applicable step by step and reaches the
goal.

In [ ]:
# ---- Self-check: is the A* plan really executable and goal-reaching? -------
s = problem.initial
for a in a_node.plan():
    assert applicable(a, s), f"the action {a} is not applicable!"
    s = result(s, a)
assert problem.is_goal(s), "the final state does not satisfy the goal!"
print("Verified: the plan is executable step by step and reaches the goal.")
print("Final state:", show_state(s))

## Reflection (briefly, in writing)
1. Why is the Sussman anomaly difficult for a planner that decomposes the goal
   into independent subgoals and solves them one after another?
2. `h_add` is **not** admissible (it can overestimate). Why does that not spoil
   the optimality of your A\* here, and when would it?
   (Hint: compare it with $h_{\max}$.)
3. How would the search change if you used **regression** (backwards) instead of
   progression? (Script part 1.3.)
4. How does `h_add` "know" anything about the domain even though it is
   domain-*independent*? (Keyword: it reads the ADD/PRE structure of the actions.)

Reference answers are at the end of the solution in the folder `solution/`.